#### [ OpenCV 워크시트 - 문제 1 (따라하기) ]

- **범__위** : 4장 이미지 처리 기초 + 6장 필터 + 모폴로지 연산
- **목__표** : 곡선 도로 이미지로 그레이스케일, HSV 변환, 블러, ROI, 캐니 엣지, 모폴로지 파이프라인을 단계별로 완성합니다.
- **이미지** : road.jpg (숲길 컬러 곡선 도로 - 흰색 실선 + 노란색 이중 차선)


In [8]:
## =======================================================
## 모듈 로딩
## =======================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
import sys
sys.path.append(r'C:\Users\rosef\OneDrive\문서\KDT-14\[7]_CVISION')
from cv_utils import *

- **[STEP 1] 이미지 데이터 준비 및 확인**

In [9]:
## =======================================================
## 모듈 로딩
## =======================================================
IMG_FILE = '../Data/images/car_load_yellow.jpg'
FILE_NAME = f'{IMG_FILE.split("/")[-1]}'

## => 이미지 로딩
imgNP = cv2.imread(IMG_FILE)
rgbNP = cv2.cvtColor(imgNP, cv2.COLOR_BGR2RGB)

## => 이미지 화면 출력
plt.show(rgbNP)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

---
- **[STEP 2] 그레이스케일 변환**

In [15]:
# 그레이스케일 변환
grayNP = cv2.cvtColor(imgNP, cv2.COLOR_BGR2GRAY)   
print(f'gray shape: {grayNP.shape}')

## => 시각화 확인 : 원본 | 그레이스케일
printPlots(1,2,['원본','그레이스케일'], [rgbNP, grayNP], [None, 'gray'])

## -------------------------------------------------------------------
## => 생각해보기: 초록 나무와 회색 도로가 그레이스케일에서 어떻게 구분되나요?
## -------------------------------------------------------------------

gray shape: (555, 948)


NameError: name 'printPlots' is not defined

---
- **[STEP 3] HSV 변환 + 노란 차선 분리**

    - 힌트
        * 노란색 차선은 그레이스케일로 변환하면 도로와 밝기가 비슷해 구분이 어렵습니다.
        * HSV의 H(색상) 채널을 이용하면 색으로 분리할 수 있습니다.

In [ ]:
## => BGR -> HSV 변환
hsvNP = cv2.cvtColor(imgNP, cv2.COLOR_BGR2HSV)  

## => 노란색 범위 지정
lower_yellow = np.array([20, 100, 100])
upper_yellow = np.array([30, 255, 255])

## => 노란색 마스크 생성
mask_yellow = cv2.inRange(hsvNP, lower_yellow, upper_yellow )   

## => 마스크를 원본에 적용
yellow_lane = cv2.bitwise_and(imgNP, imgNP, mask=mask_yellow)


## => 시각화 확인 : 원본 | 노란 차선 마스크
printPlots(1,2,['원본','노란 차선 마스크'], [rgbNP, mask_yellow], [None, 'gray'])


## -------------------------------------------------------------------
## => 생각해보기: 흰색 차선은 왜 HSV 마스크에 잡히지 않나요?
## -------------------------------------------------------------------

---
- **[STEP 4] 가우시안 블러**
    - 힌트 
        * 나무 그림자와 도로 텍스처 노이즈를 제거합니다.
        * 커널 (5,5)로 그레이스케일 이미지에 블러를 적용하세요.


In [ ]:
## => 가우시안 블러 적용
blur_5  = cv2.GaussianBlur(grayNP, (5,  5),  0)
blur_11 = cv2.GaussianBlur(grayNP, (11, 11), 0)
blur_15 = cv2.GaussianBlur(grayNP, (15, 15), 0) 

## => 시각화 확인 : 그레이스케일 | 블러(5x5) | 블러(11x11) | 블러(15x15)
printPlots( 1,4,['그레이스케일','블러(5x5)', '블러(11x11)', '블러(15x15)'], 
            [grayNP, blur_5, blur_11, blur_15], ['gray'])

## -------------------------------------------------------------------
## 생각해보기: 나무 잎사귀 텍스처가 블러 후 어떻게 변했나요?
## -------------------------------------------------------------------

---
- **[STEP 5] ROI 추출**
    - 힌트
        * 도로는 이미지 하단에 위치합니다.
        * 상단의 나무·하늘 영역을 제거하고 하단 남깁니다.
        * ★나무가 가장 적고 차선이 잘 보이는 비율을 직접 선택★


In [ ]:
h, w = blur_11.shape[:2]
print(f'이미지 크기: {w} x {h}')

# 하단 60% ROI 추출
# 힌트: y_start = h의 40% 지점
roi60 = blur_11[int(h * 0.4) : h, 0 : w]      # 하단 60%
print(f'하단 60% ROI shape: {roi60.shape}')

# 개선안 — 더 아래쪽 + 좌우도 약간 trim
roi45 = blur_11[int(h * 0.55) : h, 0 : w]     # 하단 45%
print(f'하단 45% ROI shape: {roi45.shape}')

# 추천 → 0.62 ~ 0.65 사이로 조정
roi63 = blur_11[int(h * 0.63) : h, 0 : w]
print(f'하단 37% ROI shape: {roi63.shape}')

# ROI 위치를 원본에 표시
img_box = imgNP.copy()
cv2.rectangle(img_box, (0, int(h*0.4)), (w, h), (0, 255, 0), 3)


## => 시각화 확인 : 그레이스케일 | 블러 (5x5)
printPlots(1,2,['ROI 위치 (초록 박스)','추출된 ROI'], [img_box, roi63], ['gray'])

---
- **[STEP 6] 캐니 엣지 검출**  
    - ROI 이미지에 캐니 엣지를 적용하고 블러 전/후를 비교하세요.


In [ ]:
##=> 블러 후 ROI에 캐니 엣지 적용
edges50200= cv2.Canny(roi63, 50, 200) 

##=> 블러 없이 원본 그레이 ROI에도 적용 (비교용)
gray_roi  = grayNP[int(h*0.4):h, 0:w]
edges_raw = cv2.Canny(gray_roi, 50, 150)

## => 시각화 확인 : 엣지 (블러 없음) | 엣지 (블러 5x5 후)
printPlots(1,2,['엣지 (블러 없음)','엣지 (edges50200)'], [edges_raw, edges50200], ['gray'])

## ------------------------------------------------------
## => 생각해보기: 나무 잎사귀 엣지가 블러 후 어떻게 줄었나요?
## ------------------------------------------------------

---
- **[STEP 7] 모폴로지 — 열림**
    - 힌트
        * 나무 그림자, 작은 잡음 엣지를 제거합니다.
        * 캐니 엣지 결과에 열림(Opening)을 적용하세요.


In [ ]:
## => 사각형 구조 요소 (3x3)
kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))

# 1. 먼저 팽창으로 차선을 굵게
thick = cv2.dilate(edges50200, kernel_dilate, iterations=2)

# 2. 그 다음 열림으로 노이즈 제거 : 노이즈가 있을 때 효과적
kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
result = cv2.morphologyEx(thick, cv2.MORPH_OPEN, kernel_open)

## => 시각화 확인 : 캐니 엣지 (원본) | 열림 후 (노이즈 제거)
printPlots(1,3,
           ['캐니 엣지 (원본)','thick', 'result'], 
           [edges50200, thick, result], ['gray'])

# 이미지에 따라 thick과 result가 같을 수 있음
# → 노이즈가 없는 경우 정상적인 결과

## ------------------------------------------------------
# 생각해보기: 차선 엣지는 유지되고 어떤 엣지가 제거됐나요?
## ------------------------------------------------------

---
- **[STEP 8] 모폴로지 — 닫힘**
    - 힌트:
        * 끊긴 차선 엣지를 이어줍니다.
        * 열림 결과에 닫힘(Closing)을 추가로 적용하세요.

In [ ]:

# 이 이미지는 차선이 이미 연속이라 변화 없음
# 점선 차선이나 끊긴 차선 이미지에서 효과가 나타남
# → 문제 2 (터널 이미지)에서 직접 확인 예정

## ------------------------------------------------------
## 생각해보기: 열림 -> 닫힘 순서가 바뀌면 결과가 달라질까요?
## ------------------------------------------------------

---
- **[STEP 8] 결과 저장**

In [ ]:
## => 닫힘 결과 저장
cv2.imwrite('result_road_lane.jpg', result )  

print('저장 완료: result_road_lane.jpg')

---
## 완성 체크리스트

| 단계 | 내용 | 완료 |
|------|------|------|
| STEP 1 | 그레이스케일 변환 |  |
| STEP 2 | HSV 변환 + 노란 차선 마스크 |  |
| STEP 3 | 가우시안 블러 |  |
| STEP 4 | ROI 추출 (하단 60%) |  |
| STEP 5 | 캐니 엣지 (블러 전/후 비교) |  |
| STEP 6 | 모폴로지 열림 (노이즈 제거) |  |
| STEP 7 | 모폴로지 닫힘 (차선 연결) |  |
| STEP 8 | 결과 저장 |  |
---